# Complete Test Fixture Generation

This notebook processes every MKV file in `+recordings/` and creates complete test fixtures with all pipeline outputs.

**For each recording, creates:**
- Original MKV file (copied)
- Extracted audio tracks (WAV files)
- Individual transcripts (TXT files)
- Merged transcript (JSON)
- Summary (if API key available)

**Output structure:**
```
tests/fixtures/golden_samples/
├── example_microphone/
│   ├── source.mkv
│   ├── microphone_audio.wav
│   ├── desktop_audio.wav
│   ├── microphone_transcript.txt
│   ├── desktop_transcript.txt
│   ├── merged_transcript.json
│   └── summary.txt (optional)
├── example_noisy/
│   └── ...
```

This follows the CreateTranscriptProd.ipynb pattern but creates test data for the entire test suite.

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import shutil
import json

# Add src to path so we can import minutia modules
sys.path.insert(0, str(Path('../../src').resolve()))

# Import minutia modules using the working pipeline approach
from minutia.steps import video_extraction, audio_transcription, transcript_merger, summarize_transcript
from minutia.logging import logger

# Paths
recordings_dir = Path('../../+recordings')
fixtures_dir = Path('../fixtures')
golden_samples_dir = fixtures_dir / 'golden_samples'

# Create output directory
golden_samples_dir.mkdir(parents=True, exist_ok=True)

print(f"Source recordings: {recordings_dir}")
print(f"Golden samples output: {golden_samples_dir}")

## List Available Recordings

In [ ]:
# Find all MKV files in recordings directory
mkv_files = list(recordings_dir.glob('*.mkv'))
print(f"Found {len(mkv_files)} MKV recordings:")

for mkv in mkv_files:
    size_mb = mkv.stat().st_size / (1024 * 1024)
    print(f"  - {mkv.name} ({size_mb:.1f} MB)")

if not mkv_files:
    print("⚠ No MKV files found! Make sure recordings are in +recordings/")
else:
    print(f"\nWill create {len(mkv_files)} complete fixture sets")

## Process Each Recording

Run the complete pipeline for each recording to create golden sample data

In [ ]:
def create_fixture_name(mkv_path):
    """Convert MKV filename to fixture directory name"""
    name = mkv_path.stem.lower()
    # Clean up the name for directory use
    name = name.replace('-', '_').replace('+', '_').replace(' ', '_')
    return name

def process_recording(mkv_path, output_dir):
    """Process a single recording through the complete pipeline"""
    fixture_name = create_fixture_name(mkv_path)
    fixture_dir = output_dir / fixture_name
    
    print(f"\n{'='*60}")
    print(f"Processing: {mkv_path.name} -> {fixture_name}")
    print(f"{'='*60}")
    
    # Create fixture directory
    fixture_dir.mkdir(exist_ok=True)
    
    try:
        # 1. Copy source MKV
        source_copy = fixture_dir / 'source.mkv'
        if not source_copy.exists():
            print(f"📂 Copying source MKV...")
            shutil.copy2(mkv_path, source_copy)
            print(f"   ✓ {source_copy.name}")
        else:
            print(f"📂 Source MKV already exists: {source_copy.name}")
        
        # 2. Extract audio tracks
        print(f"🎵 Extracting audio tracks...")
        result = video_extraction.process_mkv_file(str(mkv_path), str(fixture_dir))
        print(f"   ✓ Audio extraction completed")
        
        # 3. Transcribe audio files
        mic_audio = fixture_dir / 'microphone_audio.wav'
        desktop_audio = fixture_dir / 'desktop_audio.wav'
        
        if mic_audio.exists() and desktop_audio.exists():
            print(f"🎤 Transcribing audio files...")
            transcript_files = audio_transcription.process_audio_files(
                str(mic_audio), 
                str(desktop_audio), 
                output_dir=str(fixture_dir)
            )
            print(f"   ✓ Transcription completed")
            
            # 4. Merge transcripts
            print(f"🔗 Merging transcripts...")
            merged_transcript_path = fixture_dir / 'merged_transcript.json'
            transcript_json = transcript_merger.process_transcripts(
                mic_transcript_file=transcript_files['microphone'],
                desktop_transcript_file=transcript_files['desktop'],
                output_file=str(merged_transcript_path),
                parse_overlaps=False
            )
            print(f"   ✓ Transcript merging completed")
            
            # 5. Generate summary (optional - requires API key)
            try:
                print(f"📝 Generating summary...")
                transcript_data = summarize_transcript.load_transcript(str(merged_transcript_path))
                transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
                
                # Estimate cost first
                total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
                    transcript_text,
                    model='claude-3-5-sonnet'
                )
                print(f"   Estimated cost: ${estimated_price:.3f} ({total_tokens} tokens)")
                
                # Generate summary (commented out to avoid costs during fixture generation)
                # Uncomment if you want summaries in your fixtures
                # summary = summarize_transcript.generate_summary(
                #     transcript_text, 
                #     model_name='claude-3-5-sonnet'
                # )
                # summary_path = fixture_dir / 'summary.txt'
                # summarize_transcript.save_summary(summary, str(summary_path))
                # print(f"   ✓ Summary saved to {summary_path.name}")
                
                print(f"   ⚠ Summary generation skipped (uncomment to enable)")
                
            except Exception as e:
                print(f"   ⚠ Summary generation failed: {e}")
        
        else:
            print(f"   ⚠ Audio files not found, skipping transcription")
        
        # 6. Create metadata
        metadata = {
            "fixture_name": fixture_name,
            "source_file": mkv_path.name,
            "source_size_mb": round(mkv_path.stat().st_size / (1024 * 1024), 1),
            "created_files": [f.name for f in fixture_dir.glob('*') if f.is_file()],
            "description": f"Complete pipeline output for {mkv_path.name}"
        }
        
        metadata_file = fixture_dir / 'metadata.json'
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"📋 Metadata saved to {metadata_file.name}")
        print(f"✅ Fixture '{fixture_name}' completed successfully")
        
        return True
        
    except Exception as e:
        print(f"❌ Error processing {mkv_path.name}: {e}")
        logger.error(f"Fixture generation failed for {mkv_path.name}: {e}")
        return False

# Process all recordings
successful_fixtures = []
failed_fixtures = []

for mkv_file in mkv_files:
    success = process_recording(mkv_file, golden_samples_dir)
    if success:
        successful_fixtures.append(mkv_file.name)
    else:
        failed_fixtures.append(mkv_file.name)

print(f"\n{'='*60}")
print(f"FIXTURE GENERATION SUMMARY")
print(f"{'='*60}")
print(f"✅ Successful: {len(successful_fixtures)}")
for name in successful_fixtures:
    print(f"   - {name}")

if failed_fixtures:
    print(f"❌ Failed: {len(failed_fixtures)}")
    for name in failed_fixtures:
        print(f"   - {name}")
else:
    print(f"🎉 All fixtures created successfully!")

## Verify Created Fixtures

In [ ]:
# List all created fixture directories
fixture_dirs = [d for d in golden_samples_dir.iterdir() if d.is_dir()]

print(f"Created {len(fixture_dirs)} fixture directories:")
print()

total_size = 0
for fixture_dir in sorted(fixture_dirs):
    print(f"📁 {fixture_dir.name}/")
    
    # List files in this fixture
    files = list(fixture_dir.glob('*'))
    dir_size = 0
    
    for file in sorted(files):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            dir_size += size_mb
            print(f"   {file.name} ({size_mb:.1f} MB)")
    
    total_size += dir_size
    print(f"   Total: {dir_size:.1f} MB")
    print()

print(f"📊 Total fixture size: {total_size:.1f} MB")
print(f"📂 Location: {golden_samples_dir}")

## Create Master Index

Create a master metadata file listing all fixtures

In [ ]:
# Create master index of all fixtures
master_metadata = {
    "generated_by": "tests/scripts/create_complete_fixtures.ipynb",
    "description": "Complete golden sample fixtures for Minutia pipeline testing",
    "total_fixtures": len(fixture_dirs),
    "total_size_mb": round(total_size, 1),
    "fixtures": {}
}

# Load metadata from each fixture
for fixture_dir in fixture_dirs:
    metadata_file = fixture_dir / 'metadata.json'
    if metadata_file.exists():
        with open(metadata_file) as f:
            fixture_metadata = json.load(f)
        master_metadata["fixtures"][fixture_dir.name] = fixture_metadata

# Save master index
master_index_file = golden_samples_dir / 'index.json'
with open(master_index_file, 'w') as f:
    json.dump(master_metadata, f, indent=2)

print(f"📋 Master index saved to: {master_index_file}")
print(f"\nTo use these fixtures in tests:")
print(f"1. Import fixture data from {golden_samples_dir}")
print(f"2. Compare test outputs against golden samples")
print(f"3. Use for regression testing when pipeline changes")

## Next Steps

**Using these fixtures:**
1. **Unit tests** - Use individual audio/transcript files for testing components
2. **Integration tests** - Run pipeline on source MKV, compare against golden samples
3. **Regression tests** - Detect when changes break existing functionality

**Test patterns:**
```python
# Example integration test
def test_microphone_pipeline():
    fixture_dir = Path('tests/fixtures/golden_samples/example_microphone')
    source_mkv = fixture_dir / 'source.mkv'
    expected_transcript = fixture_dir / 'merged_transcript.json'
    
    # Run pipeline
    actual_output = run_pipeline(source_mkv)
    
    # Compare against golden sample
    assert_transcripts_equivalent(actual_output, expected_transcript)
```

**Maintenance:**
- Re-run this notebook when adding new recordings
- Update golden samples when intentionally changing pipeline behavior
- Keep fixture sizes reasonable for fast test execution